# Day 1: Logistic Regression from Scratch

**Goal:** Implement the mathematical foundation of logistic regression using only NumPy, before applying it to real data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import log

## The Sigmoid Function

Logistic regression maps a linear combination of inputs to a probability bounded between 0 and 1. This is achieved using the sigmoid function:

$$p = \frac{1}{1 + e^{-z}}$$

Where $z$ is the standard linear equation (with $b_0$ as bias and $b_1$ as weight):

$$z = b_0 + b_1 x$$

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z))

## Binary Cross-Entropy Loss

To evaluate how well our parameters fit the data, we calculate the log loss (negative log-likelihood). We average this across all $N$ samples:

$$\text{Loss} = - \frac{1}{N} \sum_{i=1}^{N} \left[ y_i \ln(p_i) + (1 - y_i) \ln(1 - p_i) \right]$$

*   If $y_i = 1$, the right side turns off, leaving $-\ln(p_i)$.
*   If $y_i = 0$, the left side turns off, leaving $-\ln(1 - p_i)$.

*(Note: During implementation, a small epsilon value like `1e-9` is added to the probabilities inside the logarithm to prevent a $\ln(0)$ crash.)*

In [ ]:
def binary_cross_entropy_loss(p: np.ndarray, y: np.ndarray) -> np.ndarray:
    # in numpy log defaults to ln
    loss = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))
    return loss

## Gradient Descent Optimization

To minimize the loss, we iteratively adjust the parameters by moving in the opposite direction of the gradient. 

The partial derivatives of the loss with respect to the weight ($dw$) and bias ($db$) are:

$$dw = \frac{1}{N} \sum_{i=1}^{N} x_i (p_i - y_i)$$
$$db = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)$$

The update rules, using a learning rate $\alpha$, are:

$$b_1 := b_1 - \alpha \cdot dw$$
$$b_0 := b_0 - \alpha \cdot db$$



## Derivation of the Gradient (The Chain Rule)

To find how the loss changes with respect to the weight $w$, we apply the Chain Rule:

$$\frac{d\text{Loss}}{dw} = \frac{d\text{Loss}}{dp} \cdot \frac{dp}{dz} \cdot \frac{dz}{dw}$$

### 1. Derivative of Loss with respect to $p$
Given the Loss function: 
$$\text{Loss} = - (y \ln(p) + (1 - y) \ln(1 - p))$$

We take the derivative:
$$\frac{d}{dp}[y \ln(p)] = \frac{y}{p}$$
$$\frac{d}{dp}[(1 - y)\ln(1 - p)] = -\frac{1 - y}{1 - p}$$
$$\frac{d\text{Loss}}{dp} = -\frac{y}{p} + \frac{1 - y}{1 - p}$$

### 2. Derivative of $p$ with respect to $z$
Given the Sigmoid function:
$$p = \frac{1}{1 + e^{-z}} = (1 + e^{-z})^{-1}$$

We apply the power rule and chain rule:
$$\frac{dp}{dz} = -1 \cdot (1 + e^{-z})^{-2} \cdot (-e^{-z})$$
$$\frac{dp}{dz} = \frac{e^{-z}}{(1 + e^{-z})^2} = \left(\frac{1}{1 + e^{-z}}\right) \cdot \left(\frac{e^{-z}}{1 + e^{-z}}\right)$$
$$\frac{dp}{dz} = p \cdot (1 - p)$$

### 3. Derivative of $z$ with respect to $w$
Given the linear equation:
$$z = w \cdot x + b$$
$$\frac{dz}{dw} = x$$

### 4. Combining Everything
Now we multiply the three parts together:
$$\frac{d\text{Loss}}{dw} = \left( -\frac{y}{p} + \frac{1 - y}{1 - p} \right) \cdot (p \cdot (1 - p)) \cdot x$$
$$\frac{d\text{Loss}}{dw} = (-y(1 - p) + p(1 - y)) \cdot x$$
$$\frac{d\text{Loss}}{dw} = (-y + py + p - py) \cdot x$$
$$\frac{d\text{Loss}}{dw} = (p - y) \cdot x$$

In [ ]:
def train_logistic_regression(X: np.ndarray, y: np.ndarray, learning_rate: float, epochs: int):
    # Weights and Bias start from 0
    n, f = X.shape      # n - num of samples, f - num of features 
    weights = np.zeros(f, dtype=X.dtype)  # shape (f,)
    bias = 0.0

    loss_history = list()

    for _ in range(epochs):
        # z = b1 * x + b0
        # X[N, F] * W[F, ] + B[1] = Z[N, 1]
        z = X @ weights + bias
        p = sigmoid(z)

        loss = binary_cross_entropy_loss(p, y)
        loss_history.append(loss)

        # derivative Loss(W) = x * (p - y)
        dw = (1 / n) * (X.T @ (p - y))  # [F, N] * [N, 1] = [F, 1], so for each F we get it's derivative
        db = np.mean(p - y)

        weights = weights - learning_rate * dw
        bias = bias - learning_rate * db

    return weights, bias, loss_history